# Rule of Thumb — tabular quickstart

This notebook demonstrates the tabular RoT explainer on a small synthetic
problem: we define a toy "black box" classifier, fit a Rule-of-Thumb
surrogate to its outputs, and inspect the learned feature importances.

Everything runs on CPU with synthetic data in a few seconds.

In [1]:
import numpy as np

from ruleofthumb import RuleOfThumb

In [2]:
# Synthetic data and a trivially simple "black box"
rng = np.random.RandomState(0)
n, d = 2000, 5
X = rng.randn(n, d).astype(np.float32)

def black_box(X):
    # logistic rule driven by features 0 and 2 only
    z = 2.0 * X[:, 0] - 1.5 * X[:, 2]
    return (1 / (1 + np.exp(-z)) > 0.5).astype(np.int64)  # int labels

y = black_box(X)

In [3]:
rot = RuleOfThumb(y_outputs=y, x_inputs=X, epochs=50, batch_size=500, learning_rate=0.05)
importances = rot.get_explanation(X)  # signed: positive = evidence toward class 1
importances[:3]

array([[ 2.7813909e+00,  4.2757300e-05, -1.0187041e+00, -9.6915394e-02,
         4.6934714e-03],
       [-1.3364605e+00,  1.1808680e-04,  3.9805990e-02,  9.5241528e-04,
         1.1050239e-03],
       [ 3.4791708e-01,  1.8714998e-04, -8.1479383e-01, -8.4370133e-03,
         1.1869539e-03]], dtype=float32)

In [4]:
# Global feature ranking: mean |importance| per feature (a magnitude view
# for ranking only, like SHAP summary plots — the explanations themselves
# are signed). Features 0 and 2 should dominate.
mean_imp = np.abs(importances).mean(axis=0)
for i, v in enumerate(mean_imp):
    print(f"feature {i}: {v:.4f}")

feature 0: 1.2092
feature 1: 0.0001
feature 2: 0.7477
feature 3: 0.0330
feature 4: 0.0019


## Fidelity vs. number of revealed features

`score_ordering` reveals features most-important-first and measures how well
the partially-revealed RoT reproduces the black box's labels.

In [5]:
import torch

x_t = torch.from_numpy(X)
y_t = torch.from_numpy(y.astype(np.int64))
order = rot._explainer_model.get_order(x_t)
acc = rot._explainer_model.score_ordering(x_t, y_t, order)
print("accuracy after revealing k=1..d features:")
print(acc.numpy())

accuracy after revealing k=1..d features:
[0.5035 0.9745 0.9785 0.974  0.9735 0.9735]


## Multiclass

The same pipeline works for any number of classes via `n_classes=`:
explanations come back per class (`[N, n_classes, d]`, signed), and
`score_ordering` defaults to plain accuracy, with an optional per-step
K×K confusion matrix via `return_confusion=True`.

In [6]:
# A 3-class black box driven by features 0 and 2
y3 = (X[:, 0] > 0.5).astype(np.int64) + (X[:, 2] > -0.5).astype(np.int64)  # labels in {0, 1, 2}

rot3 = RuleOfThumb(y_outputs=y3, x_inputs=X, epochs=50, batch_size=500, learning_rate=0.05, n_classes=3)
exp3 = rot3.get_explanation(X)
print("per-class explanations:", exp3.shape)  # (N, 3, d), signed

order3 = rot3._explainer_model.get_order(x_t)
acc3 = rot3._explainer_model.score_ordering(torch.from_numpy(X), torch.from_numpy(y3), order3)
print("multiclass accuracy curve:", acc3.numpy())

confusion3 = rot3._explainer_model.score_ordering(
    torch.from_numpy(X), torch.from_numpy(y3), order3, return_confusion=True
)
print("confusion after revealing all features (rows = true label):")
print(confusion3[-1].numpy())

per-class explanations: (2000, 3, 5)
multiclass accuracy curve: [0.5745 0.639  0.7945 0.7945 0.7915 0.794 ]
confusion after revealing all features (rows = true label):
[[ 297  149    0]
 [  74 1021   54]
 [   0  135  270]]
